# Hakam — full test of the final model

Runs the trained model (`refit_v3`) on all 301 test incidents, twice: every camera view (the reported protocol) and the last clip only (what the website does). Scores offence, card, body part and action family against the referee, builds the offline Arabic ruling for every answered decision, checks retrieval, and picks 10 demo videos.

**Runtime → Change runtime type → T4 or better**, then **Runtime → Run all** (≈15–25 min on GPU). Results are saved to `MyDrive/hakam_colab/full_test/`.

## 1. GPU

In [ ]:
import torch
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only - this will be slow")

## 2. Drive and code

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
!rm -rf /content/hakam && git clone -q https://github.com/FerasMad/hakam.git /content/hakam
%cd /content/hakam
!git log --oneline -1

## 3. Test clips and referee labels (test split only)

In [ ]:
import shutil, zipfile
from pathlib import Path
pkg = Path("/content/drive/MyDrive/hakam_colab")
shutil.copy(pkg / "data" / "Test.zip", "/content/Test.zip")
zipfile.ZipFile("/content/Test.zip").extractall("data/mvfouls")
labels = Path("artifacts/preprocessing/private"); labels.mkdir(parents=True, exist_ok=True)
for f in (pkg / "manifests").glob("*.csv"):
    shutil.copy(f, labels)
print(len(list(Path("data/mvfouls/Test").glob("action_*"))), "test incidents")

## 4. Install

In [ ]:
!pip install -q transformers sentence-transformers python-dotenv

## 5. Final model and the thresholds chosen on validation

In [ ]:
import json
runs = pkg / "runs"
Path("weights").mkdir(exist_ok=True)
shutil.copy(runs / "refit_v3" / "final.pt", "weights/final.pt")
drive_thresholds = json.loads((runs / "final_v3" / "metrics.json").read_text())["thresholds"]
repo_thresholds = {"offence": 0.60, "card": 0.50, "body_part": 0.49}
print("Drive  final_v3:", drive_thresholds)
print("GitHub README  :", repo_thresholds)
same = all(abs(float(drive_thresholds[k]) - v) < 1e-9 for k, v in repo_thresholds.items())
print("MATCH" if same else "MISMATCH - the README values differ from Drive; the test uses Drive")
keep = {k: float(drive_thresholds[k]) for k in ("offence", "card", "body_part")}
Path("weights/thresholds.json").write_text(json.dumps({"model_version": "hakam-refit-v3", "thresholds": keep}))

## 6. Full test

In [ ]:
!python scripts/full_test.py --weights-dir weights --out /content/drive/MyDrive/hakam_colab/full_test 2>&1 | grep -v "Warning\|warn"

## 7. Report

In [ ]:
from IPython.display import Markdown
Markdown(Path("/content/drive/MyDrive/hakam_colab/full_test/report.md").read_text())